In [0]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [0]:
store=spark.table("workspace.default.freshmart_stores")
stores=store.toPandas()
display(stores)

In [0]:
stores.shape


In [0]:
stores.dtypes


In [0]:
stores.isnull().sum()


In [0]:
stores.duplicated().sum()

In [0]:
number_of_duplicate = stores.duplicated().sum()
print(f"number_of_duplicate is {number_of_duplicate}")


In [0]:
stores.head()


In [0]:
# 1. Convert
stockouts["date"] = pd.to_datetime(stockouts["date"], errors="coerce")

# 2. Inspect
print(stockouts["date"].min())
print(stockouts["date"].max())

# 3. Check missing
print(stockouts["date"].isna().sum())

# 4. Extract
stockouts["year"] = stockouts["date"].dt.year
stockouts["month"] = stockouts["date"].dt.month
stockouts["month_name"] = stockouts["date"].dt.month_name()
stockouts["quarter"] = stockouts["date"].dt.quarter
stockouts["day"] = stockouts["date"].dt.day
stockouts["day_name"] = stockouts["date"].dt.day_name()

# 5. Analyse
stockouts.groupby("month_name").size()

In [0]:
# Read stores with correct delimiter
stores_fixed = pd.read_csv("/Volumes/workspace/default/freshmart_stores/1788896696465_freshmart_stores.csv", sep="\t")

df = transactions.merge(stores_fixed, on="store_id", how="left")
df.groupby("province").agg(
    transactions=("transaction_id","count"),
    avg_basket=("basket_value_zar","mean"),
    avg_items=("num_items","mean")
).sort_values("avg_basket", ascending=False)

In [0]:
df = transactions.merge(
    stores,
    on="store_id",
    how="left"
)
df.groupby("store_format").agg(
    avg_basket=("basket_value_zar", "mean"),
    avg_items=("num_items", "mean"),
    transactions=("transaction_id", "count")
)


In [0]:
# Get stores that have nearby competitors
competitor_stores = stores[stores["has_nearby_competitor"] == "Yes"]["store_id"].tolist()

# Filter transactions for stores with competitors
treated = df[df.store_id.isin(competitor_stores)].copy()

# Define period based on competitor open date (using median as threshold)
competitor_dates = pd.to_datetime(stores[stores["has_nearby_competitor"] == "Yes"]["competitor_open_date"].dropna())
if len(competitor_dates) > 0:
    threshold_date = competitor_dates.median()
    treated["period"] = np.where(treated.transaction_date < threshold_date, "before", "after")
    treated.groupby("period").agg(
        vol=("transaction_id", "count"),
        avg_basket=("basket_value_zar", "mean")
    )
else:
    print("No competitor open dates available for analysis")

In [0]:
# Analyze which stores drive the business (by transaction volume and revenue)
store_sales = (
    transactions
    .groupby("store_id")
    .agg(
        transaction_count=("transaction_id", "count"),
        total_revenue=("basket_value_zar", "sum"),
        avg_basket=("basket_value_zar", "mean")
    )
    .reset_index()
    .sort_values("transaction_count", ascending=False)
)

store_sales.head(10)

# Visualize top 10 stores by transaction volume
top_stores = store_sales.head(10)

plt.figure(figsize=(10, 6))

plt.barh(
    top_stores["store_id"],
    top_stores["transaction_count"]
)

plt.title("Top 10 Stores by Transaction Volume")
plt.xlabel("Number of Transactions")
plt.ylabel("Store ID")
plt.gca().invert_yaxis()

plt.show()

In [0]:
transactions = transactions.merge(
    stores[
        [
            "store_id",
            "province",
            "store_format",
            "floor_area_sqm",
            "has_nearby_competitor"
        ]
    ],
    on="store_id",
    how="left"
)
#visualise
format_analysis = (
    transactions
    .groupby(["half_year", "store_format"])
    .agg(
        avg_basket=("basket_value_zar", "mean"),
        avg_items=("num_items", "mean"),
        transactions=("transaction_id", "count")
    )
    .reset_index()
)

format_analysis